# Sales Data Analysis – Superstore Dataset

This notebook explores a US retail superstore dataset with ~9800 orders from 2015 to 2018.

I wanted to practice EDA properly — not just making random plots but actually asking specific questions first and then trying to answer them with data.

**Questions I want to answer:**
1. Which regions/states generate the most revenue?
2. Is there a seasonal pattern in sales?
3. What product categories/sub-categories perform best?
4. How do the three customer segments differ?
5. What does shipping behaviour look like?


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# consistent style across all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Loading and understanding the data

In [ ]:
df = pd.read_csv('../data/train.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# check for missing values
df.isnull().sum()

No missing values which is nice. But the date columns are strings so I need to convert them.

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

# extract some useful time columns
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['Month_Name']    = df['Order Date'].dt.strftime('%b')
df['Quarter']       = df['Order Date'].dt.quarter
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

print('Date range:', df['Order Date'].min().date(), 'to', df['Order Date'].max().date())
df[['Order Date','Ship Date','Year','Month','Shipping_Days']].head()

In [ ]:
df[['Sales','Shipping_Days']].describe().round(2)

Sales range from ~$0.44 to ~$22,638 which is a huge spread. Mean is $229 but median is probably much lower — likely some large orders pulling the average up. I'll check that.

In [ ]:
print('Median sales:', round(df['Sales'].median(), 2))
print('Mean sales:  ', round(df['Sales'].mean(), 2))
print('Orders above $1000:', (df['Sales'] > 1000).sum())

Median ($54.49) is much lower than mean ($229) — confirms a right-skewed distribution with some very large orders. That's typical for B2B datasets.

---

## 2. Revenue trends over time

In [ ]:
yearly = df.groupby('Year')['Sales'].sum().reset_index()
yearly['YoY_Growth_%'] = yearly['Sales'].pct_change() * 100
yearly['Sales_K'] = (yearly['Sales'] / 1000).round(1)
print(yearly.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# annual revenue bar chart
bars = axes[0].bar(yearly['Year'], yearly['Sales'], color=sns.color_palette('muted')[0], width=0.5)
axes[0].set_title('Annual revenue (2015–2018)')
axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, yearly['Sales']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'${val/1000:.0f}K', ha='center', fontsize=10)

# monthly revenue line — all years overlaid
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
palette = sns.color_palette('muted', 4)
for i, year in enumerate([2015, 2016, 2017, 2018]):
    monthly = df[df['Year']==year].groupby('Month_Name')['Sales'].sum().reindex(month_order)
    axes[1].plot(month_order, monthly.values, marker='o', markersize=4,
                 label=str(year), color=palette[i], linewidth=1.8)

axes[1].set_title('Monthly revenue by year')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Year', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/01_revenue_trends.png', bbox_inches='tight')
plt.show()

Revenue grew steadily from $479K in 2015 to $722K in 2018 (~57% overall). The monthly chart shows a clear pattern — **September, November, and December are consistently the strongest months** across all years. This is probably holiday/end-of-quarter purchasing.

In [ ]:
# quantify the Q4 effect
q4_share = df.groupby(['Year', 'Quarter'])['Sales'].sum().unstack()
q4_share['Q4_share_%'] = (q4_share[4] / q4_share.sum(axis=1) * 100).round(1)
print(q4_share[['Q4_share_%']])

Q4 consistently accounts for ~38–39% of annual revenue. That's a meaningful concentration — nearly 4 months worth of sales in 3 months.

---

## 3. Regional and state-level analysis

In [ ]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
print(region_sales.apply(lambda x: f'${x:,.0f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# region pie chart
colors = sns.color_palette('muted', 4)
axes[0].pie(region_sales.values, labels=region_sales.index, autopct='%1.1f%%',
            colors=colors, startangle=140, pctdistance=0.82)
axes[0].set_title('Revenue share by region')

# top 10 states
top_states = df.groupby('State')['Sales'].sum().sort_values(ascending=False).head(10)
axes[1].barh(top_states.index[::-1], top_states.values[::-1],
             color=sns.color_palette('muted')[0])
axes[1].set_title('Top 10 states by revenue')
axes[1].set_xlabel('Revenue ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for i, (state, val) in enumerate(zip(top_states.index[::-1], top_states.values[::-1])):
    axes[1].text(val + 3000, i, f'${val/1000:.0f}K', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/02_regional_breakdown.png', bbox_inches='tight')
plt.show()

West leads with 31.4% of revenue, followed closely by East (29.6%). California alone ($446K) is more than the entire South region ($389K) combined — that's striking. The top 3 states (CA, NY, TX) make up about 41% of total revenue.

---

## 4. Product category and sub-category analysis

In [ ]:
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
subcat_sales = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False)

print('Category totals:')
print(cat_sales.apply(lambda x: f'${x:,.0f}'))
print('\nTop 5 sub-categories:')
print(subcat_sales.head(5).apply(lambda x: f'${x:,.0f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# category by year grouped bar
cat_year = df.groupby(['Year','Category'])['Sales'].sum().unstack()
cat_year.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', 3), width=0.65)
axes[0].set_title('Revenue by category per year')
axes[0].set_ylabel('Revenue ($)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend(title='Category', fontsize=9)

# all sub-categories horizontal bar
colors_sub = [sns.color_palette('muted')[0] if v >= subcat_sales.median()
              else sns.color_palette('muted')[2] for v in subcat_sales.values[::-1]]
axes[1].barh(subcat_sales.index[::-1], subcat_sales.values[::-1], color=colors_sub)
axes[1].set_title('Revenue by sub-category')
axes[1].set_xlabel('Revenue ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].axvline(subcat_sales.median(), color='red', linestyle='--', linewidth=1, alpha=0.6)
axes[1].text(subcat_sales.median()+2000, 0.5, 'median', color='red', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/03_product_analysis.png', bbox_inches='tight')
plt.show()

Technology is the top category every single year. Phones and Chairs are the standout sub-categories — both around $320K. Fasteners and Labels are at the bottom (under $15K each), which makes sense as low-value consumables.

---

## 5. Customer segment analysis

In [ ]:
seg_stats = df.groupby('Segment').agg(
    Total_Revenue = ('Sales', 'sum'),
    Order_Count   = ('Order ID', 'nunique'),
    Avg_Order     = ('Sales', 'mean')
).round(2)
seg_stats['Revenue_Share_%'] = (seg_stats['Total_Revenue'] / seg_stats['Total_Revenue'].sum() * 100).round(1)
print(seg_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# segment revenue breakdown
seg_cat = df.groupby(['Segment','Category'])['Sales'].sum().unstack()
seg_cat.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', 3), width=0.6)
axes[0].set_title('Revenue by segment and category')
axes[0].set_ylabel('Revenue ($)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].legend(title='Category', fontsize=9)

# order count by segment and ship mode
seg_ship = df.groupby(['Segment','Ship Mode']).size().unstack().fillna(0)
seg_ship.plot(kind='bar', ax=axes[1], color=sns.color_palette('muted', 4), width=0.6)
axes[1].set_title('Shipping mode preference by segment')
axes[1].set_ylabel('Number of orders')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Ship Mode', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/04_segment_analysis.png', bbox_inches='tight')
plt.show()

Consumer segment dominates at 46.5% of revenue. Interestingly, all three segments follow the same category ranking (Technology > Furniture > Office Supplies) — so the category preference isn't segment-specific. Standard Class is overwhelmingly the most popular shipping mode across all segments.

---

## 6. Shipping analysis

In [ ]:
ship_stats = df.groupby('Ship Mode').agg(
    Order_Count  = ('Order ID', 'count'),
    Avg_Days     = ('Shipping_Days', 'mean'),
    Min_Days     = ('Shipping_Days', 'min'),
    Max_Days     = ('Shipping_Days', 'max')
).round(1)
ship_stats['Share_%'] = (ship_stats['Order_Count'] / ship_stats['Order_Count'].sum() * 100).round(1)
print(ship_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# shipping days distribution by mode
modes = df['Ship Mode'].unique()
palette = sns.color_palette('muted', len(modes))
for i, mode in enumerate(sorted(modes)):
    data = df[df['Ship Mode']==mode]['Shipping_Days']
    axes[0].hist(data, bins=range(0, 9), alpha=0.65, label=mode,
                 color=palette[i], edgecolor='white', density=True)
axes[0].set_title('Shipping days distribution by mode')
axes[0].set_xlabel('Days to ship')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)
axes[0].set_xticks(range(0, 8))

# order share by shipping mode
ship_counts = df['Ship Mode'].value_counts()
axes[1].pie(ship_counts.values, labels=ship_counts.index,
            autopct='%1.1f%%', colors=palette, startangle=90, pctdistance=0.82)
axes[1].set_title('Order share by shipping mode')

plt.tight_layout()
plt.savefig('../outputs/05_shipping_analysis.png', bbox_inches='tight')
plt.show()

Standard Class handles 60% of all orders but takes an average of 5 days. Same Day (as expected) ships same day or next day, but only 5.5% of customers use it — probably because of cost.

---

## 7. Sales distribution and outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# log-scale histogram of sales
axes[0].hist(df['Sales'], bins=60, color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_title('Sales distribution (raw)')
axes[0].set_xlabel('Sale amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['Sales'].mean(), color='red', linestyle='--', linewidth=1.2, label=f'Mean ${df["Sales"].mean():.0f}')
axes[0].axvline(df['Sales'].median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median ${df["Sales"].median():.0f}')
axes[0].legend(fontsize=9)

# boxplot by category
df.boxplot(column='Sales', by='Category', ax=axes[1],
           boxprops=dict(color=sns.color_palette('muted')[0]),
           flierprops=dict(marker='o', alpha=0.3, markersize=3,
                           markerfacecolor=sns.color_palette('muted')[2]))
axes[1].set_title('Sales spread by category')
axes[1].set_xlabel('')
axes[1].set_ylabel('Sale amount ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../outputs/06_sales_distribution.png', bbox_inches='tight')
plt.show()

The distribution is heavily right-skewed — most orders are small but a handful of large orders (especially in Technology) pull the mean up significantly. Technology has the widest spread and most outliers, which makes sense for high-value items like copiers and video conferencing systems.

---

## 8. Summary

In [ ]:
total_revenue = df['Sales'].sum()
total_orders  = df['Order ID'].nunique()
top_state     = df.groupby('State')['Sales'].sum().idxmax()
top_product   = df.groupby('Sub-Category')['Sales'].sum().idxmax()

print('=== Summary ===')
print(f'Total revenue (2015-2018): ${total_revenue:,.0f}')
print(f'Total unique orders:       {total_orders:,}')
print(f'Top state by revenue:      {top_state}')
print(f'Top sub-category:          {top_product}')
print(f'Revenue CAGR (2015-2018):  {((722052/479856)**(1/3)-1)*100:.1f}%')

## What I'd explore next

- This dataset doesn't include profit/margin data, which limits how useful the analysis is for business decisions. Revenue alone doesn't tell you which products are actually worth pushing.
- Would be interesting to model 2019 sales using a simple time series (even just linear regression on the monthly data).
- Customer-level analysis — repeat buyers vs one-time buyers — would require linking order IDs to customers over time.
- A geographic heatmap by state would visualise the CA/NY dominance more clearly.